In [35]:
import os
from pyspark.sql import SparkSession

# Configure local venv path
os.environ["PYSPARK_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"c:\Users\Administrator\Desktop\DE\venv\Scripts\python.exe"

spark = (
    SparkSession.builder
    .appName("Read and Filter Employees")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)

sc = spark.sparkContext

# df1 = spark.read.csv("dataset.csv", header=True, inferSchema=True)

df1 = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv("dataset.csv")  # Removed the leading /
)

# df1.createOrReplaceTempView("DataHaiBhaiIsme")


df1.createOrReplaceTempView("DataHaiBhaiIsme")

In [36]:
df1.show(50, truncate=False)

+---+-------+---------+---------+---------------+-------+---------------+
|ID | Name  |Amount($)|is Active|Order Date     |Sex    |Updated At     |
+---+-------+---------+---------+---------------+-------+---------------+
|152|Ånna   | 1 200   |Inactive |2/26/2023 22:12|FEMALE |6/1/2023 16:02 |
|415|Renée  |10.000,00|1        |1/15/2023 10:57|f      |2/13/2023 19:50|
|119|Judy   |1,000.50 |Active   |6/2/2023 17:14 |M      |4/6/2023 15:32 |
|365|Ivan   |-5000    |N        |2/2/2023 21:16 |f      |2/15/2023 21:07|
|30 |Bob    |NA       |TRUE     |NULL           |Other  |1/13/2023 1:20 |
|403|Frank  |500      |Active   |3/31/2023 0:58 |f      |4/6/2023 5:51  |
|64 |David  |2000     |NO       |1/16/2023 21:43|f      |6/4/2023 1:03  |
|482|Alice  |2.000,50 |Y        |6/10/2023 16:42|FEMALE |6/10/2023 20:17|
|442|Hannah |10.000,00|Inactive |2/8/2023 14:43 |F      |2/4/2023 7:50  |
|448|Renée  |2.000,50 |Active   |5/1/2023 17:50 |f      |1/22/2023 2:21 |
|31 |Bob    |2.000,50 |NO       |3/21/

In [37]:
import re
new_cols = []
for c in df1.columns:
    nc = re.sub(r"[\s\-]+", "_", c.strip().lower())
    nc = re.sub(r"[^0-9a-z_]+", "",nc)
    nc = re.sub(r"_+","_",nc).strip("_")
    new_cols.append(nc)
for old,new in zip(df1.columns,new_cols):
    if old != new:
        df1 = df1.withColumnRenamed(old,new)
df1.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- updated_at: string (nullable = true)



In [38]:
df1.show()

+---+-------+---------+---------+---------------+-------+---------------+
| id|   name|   amount|is_active|     order_date|    sex|     updated_at|
+---+-------+---------+---------+---------------+-------+---------------+
|152|   Ånna|    1 200| Inactive|2/26/2023 22:12| FEMALE| 6/1/2023 16:02|
|415|  Renée|10.000,00|        1|1/15/2023 10:57|      f|2/13/2023 19:50|
|119|   Judy| 1,000.50|   Active| 6/2/2023 17:14|      M| 4/6/2023 15:32|
|365|   Ivan|    -5000|        N| 2/2/2023 21:16|      f|2/15/2023 21:07|
| 30|    Bob|       NA|     TRUE|           NULL|  Other| 1/13/2023 1:20|
|403|  Frank|      500|   Active| 3/31/2023 0:58|      f|  4/6/2023 5:51|
| 64|  David|     2000|       NO|1/16/2023 21:43|      f|  6/4/2023 1:03|
|482|  Alice| 2.000,50|        Y|6/10/2023 16:42| FEMALE|6/10/2023 20:17|
|442| Hannah|10.000,00| Inactive| 2/8/2023 14:43|      F|  2/4/2023 7:50|
|448|  Renée| 2.000,50|   Active| 5/1/2023 17:50|      f| 1/22/2023 2:21|
| 31|    Bob| 2.000,50|       NO| 3/21

In [39]:
import pyspark.sql.functions as F


na_tokens = ["","na", "n/a", "none", "null", "-", "--", "unknown"]
for c,t in df1.dtypes:
    if t == 'string':
        df1 = df1.withColumn(c,F.regexp_replace(F.col(c), "\xa0", " "))
        df1 = df1.withColumn(c,F.trim(F.col(c)))
        df1 = df1.withColumn(c, F.regexp_replace(F.col(c), r"\s+", " "))
        df1 = df1.withColumn(
            c,
            F.when(F.lower(F.col(c)).isin(na_tokens), None)
            .otherwise(F.col(c))
        )

df1.show(50,truncate = False)

+---+-------+---------+---------+---------------+------+---------------+
|id |name   |amount   |is_active|order_date     |sex   |updated_at     |
+---+-------+---------+---------+---------------+------+---------------+
|152|Ånna   |1 200    |Inactive |2/26/2023 22:12|FEMALE|6/1/2023 16:02 |
|415|Renée  |10.000,00|1        |1/15/2023 10:57|f     |2/13/2023 19:50|
|119|Judy   |1,000.50 |Active   |6/2/2023 17:14 |M     |4/6/2023 15:32 |
|365|Ivan   |-5000    |N        |2/2/2023 21:16 |f     |2/15/2023 21:07|
|30 |Bob    |NULL     |TRUE     |NULL           |Other |1/13/2023 1:20 |
|403|Frank  |500      |Active   |3/31/2023 0:58 |f     |4/6/2023 5:51  |
|64 |David  |2000     |NO       |1/16/2023 21:43|f     |6/4/2023 1:03  |
|482|Alice  |2.000,50 |Y        |6/10/2023 16:42|FEMALE|6/10/2023 20:17|
|442|Hannah |10.000,00|Inactive |2/8/2023 14:43 |F     |2/4/2023 7:50  |
|448|Renée  |2.000,50 |Active   |5/1/2023 17:50 |f     |1/22/2023 2:21 |
|31 |Bob    |2.000,50 |NO       |3/21/2023 9:17 |O 

In [40]:
import pyspark.sql.functions as F

df1 = df1.withColumn("amount", F.regexp_replace("amount", "O", "0"))
df1 = df1.withColumn("amount", F.regexp_replace("amount", r"[^0-9,\.-]", ""))
df1 = df1.withColumn("amount", F.regexp_replace("amount", r"\.", ""))
df1 = df1.withColumn("amount", F.regexp_replace("amount", r",", "."))
df1 = df1.withColumn("amount", F.col("amount").cast("double"))

df1 = df1.withColumn(
    "is_active",
    F.when(F.lower(F.col("is_active")).isin(["true", "t", "yes", "y", "1", "active"]), F.lit(True))
     .when(F.lower(F.col("is_active")).isin(["false", "f", "no", "n", "o", "inactive"]), F.lit(False))
     .otherwise(F.lit(None).cast("boolean"))  # Fixed method call here
)

df1.show(10)


+---+------+-------+---------+---------------+------+---------------+
| id|  name| amount|is_active|     order_date|   sex|     updated_at|
+---+------+-------+---------+---------------+------+---------------+
|152|  Ånna| 1200.0|    false|2/26/2023 22:12|FEMALE| 6/1/2023 16:02|
|415| Renée|10000.0|     true|1/15/2023 10:57|     f|2/13/2023 19:50|
|119|  Judy| 1.0005|     true| 6/2/2023 17:14|     M| 4/6/2023 15:32|
|365|  Ivan|-5000.0|    false| 2/2/2023 21:16|     f|2/15/2023 21:07|
| 30|   Bob|   NULL|     true|           NULL| Other| 1/13/2023 1:20|
|403| Frank|  500.0|     true| 3/31/2023 0:58|     f|  4/6/2023 5:51|
| 64| David| 2000.0|    false|1/16/2023 21:43|     f|  6/4/2023 1:03|
|482| Alice| 2000.5|     true|6/10/2023 16:42|FEMALE|6/10/2023 20:17|
|442|Hannah|10000.0|    false| 2/8/2023 14:43|     F|  2/4/2023 7:50|
|448| Renée| 2000.5|     true| 5/1/2023 17:50|     f| 1/22/2023 2:21|
+---+------+-------+---------+---------------+------+---------------+
only showing top 10 

In [41]:
from pyspark.sql.functions import *

# Step 1: Trim whitespace
df1 = df1.withColumn("updated_at", trim(col("updated_at")))

# Step 2: Normalize date separators
df1 = df1.withColumn("updated_at", regexp_replace(col("updated_at"), r"[\.-]", "/"))

# Step 3: Fill missing time only if date exists
df1 = df1.withColumn(
    "updated_at",
    when(
        col("updated_at").isNotNull() & (~col("updated_at").rlike(r"\d{1,2}/\d{1,2}/\d{4}\s\d{1,2}:\d{2}")),
        concat_ws(" ", col("updated_at"), lit("00:00"))
    ).otherwise(col("updated_at"))
)

# Step 4: Safely parse timestamp (invalid formats become NULL)
df1 = df1.withColumn(
    "updated_at_ts",
    to_timestamp(col("updated_at"), "M/d/yyyy H:mm")
)

# Step 5: Format timestamp as string YYYY-MM-DD HH:MM (no seconds)
df1 = df1.withColumn(
    "updated_at",
    when(
        col("updated_at_ts").isNotNull(),
        date_format(col("updated_at_ts"), "yyyy-MM-dd HH:mm")
    ).otherwise(lit(None))
)

# Drop temporary timestamp column
df1 = df1.drop("updated_at_ts")

# df1.select("updated_at").show(10, truncate=False)
df1.show()


+---+-------+---------+---------+---------------+------+----------------+
| id|   name|   amount|is_active|     order_date|   sex|      updated_at|
+---+-------+---------+---------+---------------+------+----------------+
|152|   Ånna|   1200.0|    false|2/26/2023 22:12|FEMALE|2023-06-01 16:02|
|415|  Renée|  10000.0|     true|1/15/2023 10:57|     f|2023-02-13 19:50|
|119|   Judy|   1.0005|     true| 6/2/2023 17:14|     M|2023-04-06 15:32|
|365|   Ivan|  -5000.0|    false| 2/2/2023 21:16|     f|2023-02-15 21:07|
| 30|    Bob|     NULL|     true|           NULL| Other|2023-01-13 01:20|
|403|  Frank|    500.0|     true| 3/31/2023 0:58|     f|2023-04-06 05:51|
| 64|  David|   2000.0|    false|1/16/2023 21:43|     f|2023-06-04 01:03|
|482|  Alice|   2000.5|     true|6/10/2023 16:42|FEMALE|2023-06-10 20:17|
|442| Hannah|  10000.0|    false| 2/8/2023 14:43|     F|2023-02-04 07:50|
|448|  Renée|   2000.5|     true| 5/1/2023 17:50|     f|2023-01-22 02:21|
| 31|    Bob|   2000.5|    false| 3/21